<img src="https://pp.userapi.com/c854528/v854528797/c62ce/FiPEibYUxJc.jpg" width="40%">

## Шаг 1. Загрузка и обработка первичных данных

### Подключаем неообходимую библиотеку, читаем файл.
##### используем encoding='utf-8-sig' для удаления артефакта ∩╗┐ — это артефакт отображения BOM-метки ï»¿ (или \xef\xbb\xbf),
##### которая иногда добавляется в начало файла. Возможно она возникла при сохранении файла из excel в csv

In [2]:
import pandas as pd
data = pd.read_csv('NetflixShows.csv', encoding='utf-8-sig', sep=';')

#### Удаляем ratingDescription, т.к. он избыточен - этот столбец просто кодирует рейтинг числом у нас уже есть.

In [3]:
del data['ratingDescription']

##### и смотрим данные. 

In [4]:
data.head(3)

,title,rating,ratingLevel,release year,user rating score,user rating size
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,80
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,82
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,80


#### Удалим из данных дубликаты.
#### Дубликаты в нашем случае(а порой и не только в нашем) - это проблема качества данных.

Фактически CSV-файл содержит множество полных копий (строк) одних и тех же записей. Вот несколько причин:
1. Ошибки при парсинге, но у нас уже был готовый Excel файл, так что вина не наша )
2. Возможно, данные собирали несколько раз и просто дополняли один файл всем, что нашли
3. Или несколько человек парсили и потом "тупо" склеили в один датасет и получили, что получили. Пусть страдают другие (

#### Посмотрим глазами, чтобы убедится в наличии дублей )


In [5]:
data[data.duplicated()].sort_values('title').head(20)

,title,rating,ratingLevel,release year,user rating score,user rating size
396,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
241,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
347,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
141,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
295,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
497,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
189,13 Reasons Why,TV-MA,For mature audiences. May not be suitable for...,2017,99.0,80
201,30 Rock,TV-14,Parents strongly cautioned. May be unsuitable ...,2012,66.0,80
218,5 to 7,R,some sexual material,2014,NaN,82
265,90210,TV-14,Parents strongly cautioned. May be unsuitable ...,2013,62.0,80


#### Посмотрим сколько всего строк в файле


In [6]:
data.shape[0]

1000

#### и сколько полных дубликатов

In [7]:
data.duplicated().sum()

np.int64(500)

#### Для наглядности количество дубликатов по рейтинговым группам

In [8]:
data[data.duplicated()]['rating'].value_counts()

rating
TV-14       128
PG           94
G            85
TV-MA        66
TV-Y         32
TV-PG        26
TV-G         23
TV-Y7-FV     19
TV-Y7        15
R             5
NR            4
PG-13         3
Name: count, dtype: int64

#### И наконец-то почистим дубликаты. Используем drop_duplicates() потому что у нас задублированы целые строки, а не отдельные элементы этих строк.

In [84]:
data_no_dup = data.drop_duplicates()

## Дальше работаем с data_no_dup

In [87]:
data_no_dup.tail(5)

,title,rating,ratingLevel,release year,user rating score,user rating size
989,Russell Madness,PG,some rude humor and sports action,2015,NaN,82
993,Wiener Dog Internationals,G,General Audiences. Suitable for all ages.,2015,NaN,82
994,Pup Star,G,General Audiences. Suitable for all ages.,2016,NaN,82
997,Precious Puppies,TV-G,Suitable for all ages.,2003,NaN,82
998,Beary Tales,TV-G,Suitable for all ages.,2013,NaN,82


## И чтоже мы видим? NaN! будь он неладен.
#### И индексы "поехали"
#### Хорошо, что мы умеем и не с таким работать.

#### Пересоздадим индекс. Просто для удобства.

In [91]:
data_clear = data_no_dup.copy().reset_index(drop=True)

#### Теперь надо работать с data_clear. Разберемся с NaN. Где вообще и сколько их?

In [ ]:
data_clear.isnull().sum()

title                  0
rating                 0
ratingLevel           33
release year           0
user rating score    244
user rating size       0
dtype: int64

#### Начнем с ratingLevel. "ratingLevel" показывает описание рейтинговой группы и особенностей шоу. Т.е. мы фактически можем сами заполнить эти данные на основе того, что у нас есть.

#### Для удобства создаём словарь rating_to_level на основе существующих данных rating: ratingLevel

In [ ]:
rating_to_level = data_clear[data_clear['ratingLevel'].notna()].groupby('rating')['ratingLevel'].first().to_dict()

#### Заполняем пропуски для ratingLevel

In [92]:
data_clear['ratingLevel'] = data_clear['ratingLevel'].fillna(
    data_clear['rating'].map(rating_to_level)
)

#### Проверим, что заполнили пропуски по ratingLevel (я не параноик, я только учусь)

In [95]:
data_clear['ratingLevel'].isnull().sum()

np.int64(0)

#### Теперь приступим к "user rating score". Учитывая, что у нас 244 NaN из 500 строк - терять их не хочется. Заполним их медианой

In [96]:
data_clear['user rating score'] = data_clear.groupby('rating')['user rating score'].transform(
    lambda x: x.fillna(x.median())
)

#### Проверим, и заметим, что одна запись все еще присутствует. Она оказывается "has not been rated".

In [101]:
data_clear['user rating score'].isna().sum()

np.int64(1)

In [102]:
data_no_dup[data_no_dup['user rating score'].isna()]

,title,rating,ratingLevel,release year,user rating score,user rating size
38,White Girl,UR,This movie has not been rated. Intended for ad...,2016,NaN,82


#### Но это уже 1 из 500, просто дропнем ее и пересоздадим индекс

In [103]:
data_clear = data_clear.drop(38).reset_index(drop=True)

#### И последнее, но не в последнюю очередь )

In [105]:
data_clear.isna().sum()

title                0
rating               0
ratingLevel          0
release year         0
user rating score    0
user rating size     0
dtype: int64

In [107]:
data_clear.to_csv('NetflixShow_clear.csv', sep=';')